# **Load Parameters**


In [0]:
import traceback
try:
  catalog = dbutils.widgets.get("catalog")
  silver_schema = dbutils.widgets.get("silver_schema")
  gold_schema = dbutils.widgets.get("gold_schema")
  silver_table = dbutils.widgets.get("silver_table")
  fact_table = dbutils.widgets.get("fact_table") 
  dim_date = dbutils.widgets.get("dim_date")
  dim_vendor = dbutils.widgets.get("dim_vendor")
  dim_store = dbutils.widgets.get("dim_store")
  dim_product = dbutils.widgets.get("dim_product")
except Exception as e:
  print(f"Error retrieving widgets: {e}")
  print(traceback.format_exc())
  raise e

## **Create Fact Sales Table**

In [0]:
try:
  create_query = f"""
  CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.{fact_table} (
    -- Surrogate key for the fact table
    sale_Id BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Unique identifier for the transaction',
    
    -- Degenerate Dimension (business key)
    sale_invoice_line_no STRING,

    -- Foreign Keys
    date_key INT COMMENT 'Foreign key to dim_date',
    product_key BIGINT COMMENT 'Foreign key to dim_product',
    store_key BIGINT COMMENT 'Foreign key to dim_store',
    vendor_key BIGINT COMMENT 'Foreign key to dim_vendor',
    
    -- Measures (quantitative data)
    state_bottle_cost DECIMAL(10, 2) COMMENT 'Cost paid by the state per bottle',
    state_bottle_retail DECIMAL(10, 2) COMMENT 'Retail price paid by the store per bottle',
    bottles_sold INT COMMENT 'Number of bottles sold in the transaction',
    sale_dollars DECIMAL(10, 2) COMMENT 'Total cost of the transaction',
    volume_sold_liters DECIMAL(10, 3) COMMENT 'Total volume sold in liters',
    volume_sold_gallons DECIMAL(10, 3) COMMENT 'Total volume sold in gallons',
    
    -- Audit column
    saved_date TIMESTAMP
  )
  USING DELTA
  COMMENT 'Fact table for sales transactions'
  """
  spark.sql(create_query)
  print(f"Table {catalog}.{gold_schema}.{fact_table} ensured to exist.")

except Exception as e:
  print(f"Error creating table: {e}")
  print(traceback.format_exc())
  raise e

## **Load Data into Fact Sales Table**

In [0]:
try:
  load_query = f"""
  MERGE INTO {catalog}.{gold_schema}.{fact_table} AS target
  USING (
    -- CTE 1: Rank records from the silver source
    WITH RankedSource AS (
      SELECT
        *,
        -- Assign a rank to each record within the same invoice, ordered by date descending.
        -- 'rn = 1' will be the most recent record, handling potential duplicates.
        ROW_NUMBER() OVER(
          PARTITION BY invoice_line_no
          ORDER BY date DESC
        ) as rn
      FROM {catalog}.{silver_schema}.{silver_table}
    ),
    
    -- CTE 2: Filter to get only the deduplicated records
    DeduplicatedSource AS (
      SELECT * FROM RankedSource WHERE rn = 1
    )

    -- Final Source: Join dimensions to get foreign keys
    SELECT
      -- Degenerate Dimension
      s.invoice_line_no AS sale_invoice_line_no,

      -- Foreign Keys (resolved from dimension tables)
      d.date_key,
      p.product_key,
      st.store_key,
      v.vendor_key,
      
      -- Measures
      s.state_bottle_cost,
      s.state_bottle_retail,
      s.bottles_sold,
      s.sale_dollars,
      s.volume_sold_liters,
      s.volume_sold_gallons,
      
      -- Audit Column
      s.saved_date
      
    -- Use the deduplicated data as the source 's'
    FROM DeduplicatedSource AS s
    
    -- Join dimension tables to look up surrogate keys
    LEFT JOIN {catalog}.{gold_schema}.{dim_date} AS d
      ON s.date = d.full_date
    LEFT JOIN {catalog}.{gold_schema}.{dim_product} AS p
      ON s.item_number = p.product_business_key
    LEFT JOIN {catalog}.{gold_schema}.{dim_store} AS st
      ON s.store_number = st.store_business_key
    LEFT JOIN {catalog}.{gold_schema}.{dim_vendor} AS v
      ON s.vendor_number = v.vendor_business_key
  ) AS source
  -- Define the business key for the MERGE operation
  ON 
    target.sale_invoice_line_no = source.sale_invoice_line_no

  WHEN MATCHED THEN
    -- Update all fields in case of re-processing
    UPDATE SET
      target.date_key = COALESCE(source.date_key, -1),
      target.product_key = COALESCE(source.product_key, -1),
      target.store_key = COALESCE(source.store_key, -1),
      target.vendor_key = COALESCE(source.vendor_key, -1),
      target.state_bottle_cost = source.state_bottle_cost,
      target.state_bottle_retail = source.state_bottle_retail,
      target.bottles_sold = source.bottles_sold,
      target.sale_dollars = source.sale_dollars,
      target.volume_sold_liters = source.volume_sold_liters,
      target.volume_sold_gallons = source.volume_sold_gallons,
      target.saved_date = source.saved_date

  WHEN NOT MATCHED THEN
    -- Insert new transaction record
    INSERT (
      sale_invoice_line_no,
      date_key,
      product_key,
      store_key,
      vendor_key,
      state_bottle_cost,
      state_bottle_retail,
      bottles_sold,
      sale_dollars,
      volume_sold_liters,
      volume_sold_gallons,
      saved_date
    )
    VALUES (
      source.sale_invoice_line_no,
      -- Use -1 for unmatched dimensions (e.g., 'Unknown' member)
      COALESCE(source.date_key, -1),
      COALESCE(source.product_key, -1),
      COALESCE(source.store_key, -1),
      COALESCE(source.vendor_key, -1),
      source.state_bottle_cost,
      source.state_bottle_retail,
      source.bottles_sold,
      source.sale_dollars,
      source.volume_sold_liters,
      source.volume_sold_gallons,
      source.saved_date
    )
  """
  spark.sql(load_query)
  print(f"Successfully merged data into {catalog}.{gold_schema}.{fact_table}.")

except Exception as e:
  print(f"Error merging data into fact table: {e}")
  print(traceback.format_exc())
  raise e